# FlatBusNet + IdealBusNet — the 12-run comparison at 100k

Both private-module bus models in one notebook, full budget.

**FlatBusNet** — CNN, flatten, each private module projects the whole
flattened image through its own linear map (per-position weights, no
pooling anywhere), then the unchanged bus. The escaper-trait test.

**IdealBusNet** — canonical competitive field binding with per-module
learned anchor priors: identity, binding and addressing in one phase-space
geometry; private cells, shared protocol. Set both flags off and the
canonical model returns.

| family | arms |
|---|---|
| `flat` | `private`, `shared-gru`, `static` |
| `ideal` | `ideal`, `shared-anchors`, `shared-gru` |

Six arms × two tasks = 12 runs at 100k steps, ~25–40 min each compiled on a
4090 (SoC ≈ 3–4 h per family block, SQOOP similar). To use both GPUs, run
the SoC section in one kernel and the SQOOP section in another
(`CUDA_VISIBLE_DEVICES=0` / `1`); results dicts are independent.
Interventions: `freeze` / `shuffle` for both families, plus
`anchor_shuffle` (permute identity priors) for `ideal`.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'main.py').exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.core.config import Config, TrainConfig, LoggingConfig, OptimConfig
from src.tasks.sort_of_clevr.config import SortOfClevrDataConfig
from src.tasks.sqoop.config import SqoopDataConfig
from src.tasks.sort_of_clevr.callbacks import AccuracyCallbackCfg, QtypeAccuracyCallbackCfg
from src.tasks.sqoop.callbacks.metrics import SqoopAccuracyCallbackCfg
from src.tasks import TASKS


In [2]:
# THE SIMPLEST PRIVATE-MODULE BUS MODEL.
#
# CNN -> flatten -> each of M PRIVATE modules projects the whole flattened
# image through its own linear map (conv_lstm's position-indexed weights,
# per module), plus its own view of the question. No field, no slots, no
# attention, no pooling anywhere. Then T steps of the unchanged bus:
# message m_i = W_m h_i in R^K, phase z_i on S^{D-1}, wire carries
# sum_j m_j z_j^T, each row receives through its own frame, echo cancelled,
# private GRU cells update, phases advance (rotation + tanh-MLP coupling +
# stimulus W_s h). A head holds only the question and answers alone.
#
# flags: private=False shares one GRU cell; static=True fixes learned
#        addresses with no dynamics; forward(..., phase_override=
#        'freeze'|'shuffle') for the interventions.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

M, D, K, DM, T, DT = 6, 6, 4, 96, 8, 0.5      # modules, phase dim, message dim, state, steps
FLAT_CH, HID = 16, 128                         # channels kept at the flatten; readout width


def tang(z, v):
    return v - (v * z).sum(-1, keepdim=True) * z


class FlatBusNet(nn.Module):

    def __init__(self, img_size: int, q_size: int, answer_dim: int,
                 private: bool = True, static: bool = False):
        super().__init__()
        self.q_size, self.private, self.static, self.N = q_size, private, static, M + 1
        ch, s = 48, img_size
        self.enc = nn.Sequential(
            nn.Conv2d(3, ch, 3, 2, 1), nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, 2, 1), nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, FLAT_CH, 3, 1, 1),
        )
        s = (s + 3) // 4
        flat = FLAT_CH * s * s
        self.proj = nn.ModuleList([nn.Linear(flat, DM) for _ in range(M)])   # private view of the image
        self.qin = nn.ModuleList([nn.Linear(q_size, DM) for _ in range(M)])  # private view of the question
        self.head_init = nn.Sequential(nn.Linear(q_size, 64), nn.GELU(), nn.Linear(64, DM))
        mk = lambda: nn.GRUCell(K * D, DM)
        self.cells = nn.ModuleList([mk() for _ in range(self.N)]) if private else mk()
        self.msg = nn.Linear(DM, K)                                          # the n -> k projection
        g = torch.Generator().manual_seed(1234)
        self.register_buffer('ref', F.normalize(torch.randn(D - 1, D, generator=g), dim=-1))
        self.z0 = nn.Parameter(F.normalize(torch.randn(self.N, D), dim=-1))  # learned starting addresses
        self.omega = nn.Parameter(torch.zeros(self.N))
        self.Kc = nn.Parameter(torch.ones(self.N, self.N))
        self.kmlp = nn.Sequential(nn.Linear(2 * DM, 64), nn.GELU(), nn.Linear(64, 1), nn.Tanh())
        self.zstim = nn.Linear(DM, D)
        self.A = nn.Parameter(0.1 * torch.randn(D, D))
        self.out = nn.Sequential(nn.Linear(DM + q_size, HID), nn.GELU(), nn.Linear(HID, answer_dim))
        self.prior = nn.Sequential(nn.Linear(q_size, HID), nn.GELU(), nn.Linear(HID, answer_dim))

    def _frame(self, z):
        vecs = [z]
        for k in range(D - 1):
            v = self.ref[k].to(z.dtype).expand_as(z)
            for u in vecs:
                v = v - (v * u).sum(-1, keepdim=True) * u
            vecs.append(F.normalize(v, dim=-1))
        return torch.stack(vecs, 2)

    def _receive(self, h, z):
        m = self.msg(h)
        bus = torch.einsum('bnK,bnd->bKd', m, z)
        Fr = self._frame(z)
        r = torch.einsum('bKd,bnad->bnKa', bus, Fr) - torch.einsum('bnK,bnd,bnad->bnKa', m, z, Fr)
        return r.flatten(2) / float(self.N)

    def _zstep(self, z, h):
        B = h.shape[0]
        A = self.A - self.A.t()
        A = A / (A.norm() / math.sqrt(2) + 1e-6)
        vel = self.omega.to(z.dtype)[None, :, None] * torch.einsum('de,bne->bnd', A, z)
        hi = h.unsqueeze(2).expand(B, self.N, self.N, DM)
        hj = h.unsqueeze(1).expand(B, self.N, self.N, DM)
        kap = self.kmlp(torch.cat([hi, hj], -1)).squeeze(-1)
        vel = vel + tang(z, torch.einsum('bij,bjd->bid', self.Kc.to(z.dtype)[None] * kap, z))
        vel = vel + tang(z, self.zstim(h))
        return F.normalize(z + DT * vel, dim=-1)

    def forward(self, images, q, phase_override=None):
        B = images.shape[0]
        q = q.float()
        flat = self.enc(images).flatten(1)
        h = torch.stack([self.proj[k](flat) + self.qin[k](q) for k in range(M)], 1)
        h = torch.cat([h, self.head_init(q).unsqueeze(1)], 1)
        z = F.normalize(self.z0, dim=-1)[None].expand(B, -1, -1).to(h.dtype)
        if phase_override == 'shuffle' and not self.static:
            perm = torch.randperm(M, device=h.device)
            z = torch.cat([z[:, perm], z[:, M:]], 1)
        for _ in range(T):
            r = self._receive(h, z)
            if self.private:
                h = torch.stack([self.cells[k](r[:, k], h[:, k]) for k in range(self.N)], 1)
            else:
                h = self.cells(r.reshape(B * self.N, -1), h.reshape(B * self.N, DM)).reshape(B, self.N, DM)
            if not self.static and phase_override != 'freeze':
                z = self._zstep(z, h)
        logits = self.out(torch.cat([h[:, M], q], -1)) + self.prior(q)
        return {'logits': logits, 'z': z}


In [3]:
# THE PROPOSED MODEL -- private computation, shared protocol, identity in
# phase space.
#
# Binding is the canonical competitive claim (softmax over SLOTS, mean-shift,
# exclusivity: every cell gets one owner), with one change: each private
# module owns its OWN learned anchor distribution -- its prior over where in
# phase space it starts hunting. Identity therefore lives in the same
# geometry as binding and addressing: module k habitually wins the phase
# neighbourhoods its prior prefers, so its private GRU cell and identity
# embedding have a stable referent to specialise against, with no ordering
# convention and no partition. The slot's bus phase comes from its anchor
# (who you are sets where you broadcast from). Everything that constitutes
# the PROTOCOL stays shared: message projection, stimulus map, kappa-MLP,
# tokenisation -- private minds, one radio standard. Bus, dynamics, head
# and prior are canonical.
#
# flags: per_module_anchors=False -> one shared anchor distribution;
#        private=False -> one shared GRU cell; static=True -> learned fixed
#        addresses, no dynamics. forward(..., phase_override=) supports
#        'freeze', 'shuffle' (permute addresses) and 'anchor_shuffle'
#        (permute the anchor priors: routing-by-name vs routing-by-address).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

M, D, DM, MSG, T, DT = 6, 6, 96, 4, 8, 0.5
FCH, FG, FD, TF = 64, 16, 4, 8
TOK, HID, ITERS = 64, 128, 3


def tang(z, v):
    return v - (v * z).sum(-1, keepdim=True) * z


class Field(nn.Module):
    def __init__(self):
        super().__init__()
        C = FG * FD
        self.z_head = nn.Conv2d(FCH, C, 1)
        self.stim = nn.Conv2d(FCH, C, 1)
        self.J = nn.Conv2d(C, C, 5, padding=2, bias=False)
        nn.init.normal_(self.J.weight, std=0.02)
        self.omega = nn.Parameter(torch.randn(FG, FD, FD) * 0.1)

    def _norm(self, z):
        B, C, S, _ = z.shape
        return F.normalize(z.view(B, FG, FD, S, S), dim=2).view(B, C, S, S)

    def forward(self, f):
        z = self._norm(self.z_head(f))
        c = self.stim(f)
        A = (self.omega - self.omega.transpose(-1, -2)) * 0.1
        B, C, S, _ = z.shape
        for _ in range(TF):
            zg = z.view(B, FG, FD, S, S)
            rot = torch.einsum('kde,bkeij->bkdij', A, zg).reshape(B, C, S, S)
            dg = (self.J(z) + c).view(B, FG, FD, S, S)
            drv = (dg - (dg * zg).sum(2, keepdim=True) * zg).reshape(B, C, S, S)
            z = self._norm(z + rot + drv)
        return z.view(B, FG, FD, S * S).permute(0, 3, 1, 2)


class IdealBusNet(nn.Module):

    def __init__(self, img_size: int, q_size: int, answer_dim: int,
                 private: bool = True, per_module_anchors: bool = True, static: bool = False):
        super().__init__()
        self.q_size, self.private, self.static, self.N = q_size, private, static, M + 1
        layers, cin, s = [], 3, img_size
        for _ in range(2):
            layers += [nn.Conv2d(cin, 48, 3, 2, 1), nn.GroupNorm(8, 48), nn.SiLU()]
            cin, s = 48, (s + 1) // 2
        layers += [nn.Conv2d(48, 48, 3, 1, 1), nn.GroupNorm(8, 48), nn.SiLU(), nn.Conv2d(48, FCH, 3, 1, 1)]
        self.enc = nn.Sequential(*layers)
        self.g_gamma, self.g_beta = nn.Linear(q_size, FCH), nn.Linear(q_size, FCH)
        self.gn = nn.GroupNorm(8, FCH)
        self.pos = nn.Parameter(0.02 * torch.randn(1, FCH, s, s))
        self.field = Field()
        A_M = M if per_module_anchors else 1
        self.anchor_mu = nn.Parameter(torch.randn(1, A_M, FG, FD))          # identity as a phase-space prior
        self.anchor_ls = nn.Parameter(torch.zeros(1, A_M, FG, FD))
        self.log_beta = nn.Parameter(torch.log(torch.tensor(8.0)))
        self.to_tok = nn.Sequential(nn.LayerNorm(FCH), nn.Linear(FCH, TOK))
        self.f2g, self.f2b = nn.Linear(q_size, TOK), nn.Linear(q_size, TOK)
        self.tnorm = nn.LayerNorm(TOK)
        self.e = nn.Parameter(torch.randn(M, DM) / DM ** 0.5)
        self.h_init = nn.Sequential(nn.Linear(q_size, 64), nn.GELU(), nn.Linear(64, M * DM))
        self.head_init = nn.Sequential(nn.Linear(q_size, 64), nn.GELU(), nn.Linear(64, DM))
        mk = lambda: nn.GRUCell(TOK + MSG * D, DM)
        self.cells = nn.ModuleList([mk() for _ in range(self.N)]) if private else mk()
        self.msg = nn.Linear(DM, MSG)                                       # shared protocol from here down
        g = torch.Generator().manual_seed(1234)
        self.register_buffer('ref', F.normalize(torch.randn(D - 1, D, generator=g), dim=-1))
        self.az = nn.Linear(FG * FD, D)
        if static:
            self.z_static = nn.Parameter(F.normalize(torch.randn(self.N, D), dim=-1))
        self.omega = nn.Parameter(torch.zeros(self.N))
        self.Kc = nn.Parameter(torch.ones(self.N, self.N))
        self.kmlp = nn.Sequential(nn.Linear(2 * DM, 64), nn.GELU(), nn.Linear(64, 1), nn.Tanh())
        self.zstim = nn.Linear(DM, D)
        self.A = nn.Parameter(0.1 * torch.randn(D, D))
        self.out = nn.Sequential(nn.Linear(DM + q_size, HID), nn.GELU(), nn.Linear(HID, answer_dim))
        self.prior = nn.Sequential(nn.Linear(q_size, HID), nn.GELU(), nn.Linear(HID, answer_dim))

    def _frame(self, z):
        vecs = [z]
        for k in range(D - 1):
            v = self.ref[k].to(z.dtype).expand_as(z)
            for u in vecs:
                v = v - (v * u).sum(-1, keepdim=True) * u
            vecs.append(F.normalize(v, dim=-1))
        return torch.stack(vecs, 2)

    def _receive(self, h, z):
        m = self.msg(h)
        bus = torch.einsum('bnD,bnd->bDd', m, z)
        Fr = self._frame(z)
        r = torch.einsum('bDd,bnad->bnDa', bus, Fr) - torch.einsum('bnD,bnd,bnad->bnDa', m, z, Fr)
        return r.flatten(2) / float(self.N)

    def _zstep(self, z, h):
        B = h.shape[0]
        A = self.A - self.A.t()
        A = A / (A.norm() / math.sqrt(2) + 1e-6)
        vel = self.omega.to(z.dtype)[None, :, None] * torch.einsum('de,bne->bnd', A, z)
        hi = h.unsqueeze(2).expand(B, self.N, self.N, DM)
        hj = h.unsqueeze(1).expand(B, self.N, self.N, DM)
        kap = self.kmlp(torch.cat([hi, hj], -1)).squeeze(-1)
        vel = vel + tang(z, torch.einsum('bij,bjd->bid', self.Kc.to(z.dtype)[None] * kap, z))
        vel = vel + tang(z, self.zstim(h))
        return F.normalize(z + DT * vel, dim=-1)

    def forward(self, images, q, phase_override=None):
        B = images.shape[0]
        q = q.float()
        f = self.enc(images)
        f = f * (1 + self.g_gamma(q))[..., None, None] + self.g_beta(q)[..., None, None]
        f = self.gn(f) + self.pos
        Zt = self.field(f)
        feats = f.flatten(2).transpose(1, 2)
        mu, ls = self.anchor_mu.expand(1, M, FG, FD), self.anchor_ls.expand(1, M, FG, FD)
        if phase_override == 'anchor_shuffle':
            perm = torch.randperm(M, device=f.device)
            mu, ls = mu[:, perm], ls[:, perm]
        phi = F.normalize(mu + ls.exp() * torch.randn(B, M, FG, FD, device=f.device, dtype=f.dtype), dim=-1)
        reads = None
        for _ in range(ITERS):                                              # canonical competition: cells choose slots
            logits = self.log_beta.exp() * torch.einsum('bmgd,bpgd->bmp', phi, Zt) / FG
            attn = F.softmax(logits, dim=1)
            reads = attn / (attn.sum(-1, keepdim=True) + 1e-8)
            phi = F.normalize(torch.einsum('bmp,bpgd->bmgd', reads, Zt), dim=-1)
        X = self.tnorm(self.to_tok(torch.einsum('bmp,bpf->bmf', reads, feats))
                       * (1 + self.f2g(q)).unsqueeze(1) + self.f2b(q).unsqueeze(1))
        X = torch.cat([X, torch.zeros(B, 1, TOK, device=X.device, dtype=X.dtype)], 1)
        h = torch.cat([self.h_init(q).reshape(B, M, DM) + self.e[None],
                       self.head_init(q).unsqueeze(1)], 1)
        if self.static:
            z = F.normalize(self.z_static, dim=-1)[None].expand(B, -1, -1).to(X.dtype)
        else:
            zs = F.normalize(self.az(phi.flatten(2)), dim=-1)
            if phase_override == 'shuffle':
                zs = zs[:, torch.randperm(M, device=X.device)]
            zh = F.normalize(torch.randn(B, 1, D, device=X.device, dtype=X.dtype), dim=-1)
            z = torch.cat([zs, zh], 1)
        for _ in range(T):
            r = self._receive(h, z)
            inp = torch.cat([X, r], -1)
            if self.private:
                h = torch.stack([self.cells[k](inp[:, k], h[:, k]) for k in range(self.N)], 1)
            else:
                h = self.cells(inp.reshape(B * self.N, -1), h.reshape(B * self.N, DM)).reshape(B, self.N, DM)
            if not self.static and phase_override != 'freeze':
                z = self._zstep(z, h)
        logits = self.out(torch.cat([h[:, M], q], -1)) + self.prior(q)
        return {'logits': logits, 'read_attn': reads, 'z': z}


In [4]:
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration

from src import build_dataloaders, build_optim, build_lr_scheduler, build_loss_fn, build_callbacks
from src.training import Trainer
from src.training.utils import set_seed


class OnTask(nn.Module):
    """SoC questions are 18-d vectors; SQOOP questions are 3 ints one-hot to 120-d."""

    def __init__(self, core: nn.Module, kind: str):
        super().__init__()
        self.core, self.kind = core, kind

    def forward(self, batch, **kw):
        if self.kind == 'soc':
            q = batch['questions'].float()
        else:
            q = F.one_hot(batch['questions'].long(), 40).float().flatten(1)
        return self.core(batch['images'], q, phase_override=kw.get('phase_override'))


def train(model: nn.Module, cfg: Config, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    set_seed(cfg.train.seed)
    accelerator = Accelerator(
        mixed_precision=cfg.train.mixed_precision,
        gradient_accumulation_steps=cfg.train.grad_accum,
        project_config=ProjectConfiguration(project_dir=out_dir),
    )
    try:
        if cfg.train.compile_model:
            model.compile()
        optimiser = build_optim(model, cfg.optim)
        trainer = Trainer(
            cfg=cfg, out_dir=out_dir, logger=None, model=model,
            dataloaders=build_dataloaders(cfg, str(accelerator.device)),
            optimiser=optimiser,
            scheduler=build_lr_scheduler(optimiser, cfg.train.n_steps, cfg.optim),
            accelerator=accelerator, callbacks=build_callbacks(cfg),
            loss_fn=build_loss_fn(cfg),
        )
        return trainer.train(), trainer
    finally:
        accelerator.end_training()


def run_families(families, cfg, out_root, kind, img_size, q_size, answer_dim):
    results, models = {}, {}
    for fam, (cls, arms) in families.items():
        for arm, kw in arms.items():
            name = f'{fam}/{arm}'
            model = OnTask(cls(img_size, q_size, answer_dim, **kw), kind)
            print(f'\n===== {name}  ({sum(p.numel() for p in model.parameters()):,} params) =====')
            results[name], _ = train(model, cfg, str(out_root / name.replace('/', '_')))
            models[name] = model
    return results, models


def intervene(model, loader, ovs, n_batches=4):
    model.eval(); out = {}
    with torch.no_grad():
        for ov in ovs:
            n = c = 0
            for i, b in enumerate(loader):
                if i == n_batches: break
                p = model(b, phase_override=ov)['logits'].argmax(-1)
                c += (p == b['answers']).sum().item(); n += len(p)
            out[ov or 'none'] = c / max(n, 1)
    return out


# FAMILIES = {
#     'flat':  (None, {'private': dict(), 'shared-gru': dict(private=False), 'static': dict(static=True)}),
#     'ideal': (None, {'ideal': dict(), 'shared-anchors': dict(per_module_anchors=False), 'shared-gru': dict(private=False)}),
# }
# OVS = {'flat': [None, 'freeze', 'shuffle'], 'ideal': [None, 'freeze', 'shuffle', 'anchor_shuffle']}
# STEPS = 100_000

FAMILIES = {
    'ideal': (None, {'ideal': dict()}),
}
OVS = {'ideal': [None, 'freeze', 'shuffle', 'anchor_shuffle']}
STEPS = 200_000

In [5]:
# FAMILIES['flat'] = (FlatBusNet, FAMILIES['flat'][1])
FAMILIES['ideal'] = (IdealBusNet, FAMILIES['ideal'][1])

OUT = ROOT / 'notebooks' / 'outputs' / 'privbus_100k'


def mk_cfg(ds, callbacks):
    return Config(
        train=TrainConfig(seed=0, n_steps=STEPS, train_bs=256, val_bs=1024,
                          early_stop_metric='loss', early_stop_big_is_better=False,
                          early_stop_patience=10**6, early_stop_min_delta=0.0,
                          mixed_precision='bf16', compile_model=True,
                          grad_accum=1, grad_clip=1.0, loader_mode='gpu_cached', num_workers=0),
        logging=LoggingConfig(eval_log_interval=2000, train_log_interval=500,
                              info_metrics=['loss', 'accuracy'], save_best=False),
        optim=OptimConfig(optimiser='adamw', lr=3e-4, weight_decay=0.01,
                          lr_scheduler='warmup_cosine', lr_scheduler_params={'warmup_steps': 2000}),
        dataset=ds, callbacks=callbacks,
    )


In [6]:
soc_ds = SortOfClevrDataConfig(
    name='sort_of_clevr', seed=1, root=str(ROOT / 'data'), dir='sort-of-clevr-nb-36k',
    train_size=36_000, test_size=1000, img_size=75, obj_size=5, nb_questions=10, t_subtype=-1,
)
if not (Path(soc_ds.root) / soc_ds.dir).exists():
    TASKS['sort_of_clevr'].prepare(soc_ds)

soc_cfg = mk_cfg(soc_ds, [AccuracyCallbackCfg(), QtypeAccuracyCallbackCfg()])
soc_results, soc_models = run_families(FAMILIES, soc_cfg, OUT / 'soc', 'soc', 75, 18, 10)



===== ideal/ideal  (670,434 params) =====
Starting training
Model: OnTask
Parameters: total=670434
Config:
train:
  seed: 0
  n_steps: 200000
  train_bs: 256
  val_bs: 1024
  early_stop_metric: loss
  early_stop_big_is_better: false
  early_stop_patience: 1000000
  early_stop_min_delta: 0.0
  mixed_precision: bf16
  compile_model: true
  grad_accum: 1
  grad_clip: 1.0
  loader_mode: gpu_cached
  num_workers: 0
logging:
  eval_log_interval: 2000
  train_log_interval: 500
  info_metrics:
  - loss
  - accuracy
  save_best: false
wandb:
  enabled: false
  project_name: null
  entity: null
  tags: []
  run_name: null
optim:
  optimiser: adamw
  lr: 0.0003
  weight_decay: 0.01
  lr_scheduler: warmup_cosine
  lr_scheduler_params:
    warmup_steps: 2000
dataset:
  name: sort_of_clevr
  root: /home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data
  dir: sort-of-clevr-nb-36k
  seed: 1
  train_size: 36000
  test_size: 1000
  img_size: 75
  obj_size: 5
  nb_questions: 10
  t_subtype: -1

In [7]:
FLOORS = {'accuracy': 0.492, 'binary_accuracy': 0.433, 'ternary_accuracy': 0.538}

print(f"{'run':<22}" + ''.join(f"{k.split('_')[0]:>10}" for k in FLOORS)
      + f"{'freeze':>9}{'shuffle':>9}{'a_shuf':>9}")
soc_test_loader = build_dataloaders(soc_cfg, 'cuda' if torch.cuda.is_available() else 'cpu')[2]
for name, model in soc_models.items():
    fam = name.split('/')[0]
    r = soc_results[name]
    iv = intervene(model, soc_test_loader, OVS[fam])
    row = ''.join(f"{r['callbacks/' + k] - v:>+10.3f}" for k, v in FLOORS.items())
    row += ''.join(f"{iv['none'] - iv[k]:>+9.3f}" for k in ['freeze', 'shuffle'])
    row += f"{iv['none'] - iv['anchor_shuffle']:>+9.3f}" if 'anchor_shuffle' in iv else f"{'---':>9}"
    print(f'{name:<22}' + row)

print('\ndeltas above the question-only floors; drops = accuracy cost of the intervention.')
print('shuffle = routing-by-address; a_shuf = routing-by-name (ideal family only).')

run                     accuracy    binary   ternary   freeze  shuffle   a_shuf
ideal/ideal               +0.294    +0.518    +0.037   +0.404   -0.005   +0.047

deltas above the question-only floors; drops = accuracy cost of the intervention.
shuffle = routing-by-address; a_shuf = routing-by-name (ideal family only).


In [8]:
sq_ds = SqoopDataConfig(
    name='sqoop', seed=0, root=str(ROOT / 'data'),
    dir='sqoop-seed0-train1080000-test25600-rhs18-img64-objs5-minobj10-maxobj15',
    train_size=1_080_000, test_size=25_600, rhs_variety=18,
)
sq_cfg = mk_cfg(sq_ds, [SqoopAccuracyCallbackCfg()])
sq_results, sq_models = run_families(FAMILIES, sq_cfg, OUT / 'sqoop', 'sqoop', 64, 120, 2)



===== ideal/ideal  (726,930 params) =====


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.34 GiB. GPU 0 has a total capacity of 23.52 GiB of which 11.72 GiB is free. Process 2399833 has 7.21 GiB memory in use. Process 2752726 has 3.80 GiB memory in use. Including non-PyTorch memory, this process has 778.00 MiB memory in use. Of the allocated memory 180.07 MiB is allocated by PyTorch, and 103.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
print(f"{'run':<22}{'test':>8}{'freeze':>9}{'shuffle':>9}{'a_shuf':>9}   (floor = .500)")
sq_test_loader = build_dataloaders(sq_cfg, 'cuda' if torch.cuda.is_available() else 'cpu')[2]
for name, model in sq_models.items():
    fam = name.split('/')[0]
    r = sq_results[name]
    iv = intervene(model, sq_test_loader, OVS[fam])
    row = f"{r['callbacks/accuracy']:>8.3f}"
    row += ''.join(f"{iv['none'] - iv[k]:>+9.3f}" for k in ['freeze', 'shuffle'])
    row += f"{iv['none'] - iv['anchor_shuffle']:>+9.3f}" if 'anchor_shuffle' in iv else f"{'---':>9}"
    print(f'{name:<22}' + row)

print('\ntest is held-out pairs. flat/private is the escaper-trait test (no pooling')
print('anywhere): watch its train CE against .693. ideal pools at the read, so chance')
print('there keeps the diagnosis intact; any escape is front-page news.')